
# Trabajo Final - Análisis de Aerolíneas

## Apartado 1. Ingestión y limpieza básica (Bronze)

En esta etapa se cargan los archivos CSV de vuelos, aerolíneas y aeropuertos. Se corrigen los tipos de datos, se eliminan registros sin información obligatoria y se almacenan los resultados en tablas Delta Lake.


In [0]:

# 1. Importar librerias

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()


# 2. Configuracion del proyecto

CATALOG = "trabajo_final_aerolineas"
BRONZE_SCHEMA = "bronze"
VOLUME_NAME = "landing"

BASE_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"

FLIGHTS_PATH = f"{BASE_PATH}/flights.csv"
AIRLINES_PATH = f"{BASE_PATH}/airlines.csv"
AIRPORTS_PATH = f"{BASE_PATH}/airports.csv"


# 3. Crear catalogo, esquema y volumen

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
spark.sql(
    f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}"
)


# 4. Cargar archivos CSV

flights_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(FLIGHTS_PATH)
)

airlines_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(AIRLINES_PATH)
)

airports_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(AIRPORTS_PATH)
)


# 5. Limpiar datos de vuelos

# Ajustar tipos de datos y quitar registros incompletos

flights_bronze = (
    flights_raw
    .withColumn(
        "flight_id",
        F.trim(F.col("flight_id").cast("string"))
    )
    .withColumn(
        "flight_date",
        F.to_date(F.col("flight_date"))
    )
    .withColumn(
        "airline_id",
        F.trim(F.col("airline_id").cast("string"))
    )
    .withColumn(
        "origin_airport",
        F.trim(F.col("origin_airport").cast("string"))
    )
    .withColumn(
        "destination_airport",
        F.trim(F.col("destination_airport").cast("string"))
    )
    .withColumn(
        "departure_delay",
        F.col("departure_delay").cast("int")
    )
    .withColumn(
        "arrival_delay",
        F.col("arrival_delay").cast("int")
    )
    .withColumn(
        "distance_miles",
        F.col("distance_miles").cast("double")
    )
    .filter(
        F.col("flight_id").isNotNull()
        & (F.col("flight_id") != "")
        & F.col("airline_id").isNotNull()
        & (F.col("airline_id") != "")
        & F.col("origin_airport").isNotNull()
        & (F.col("origin_airport") != "")
    )
    .withColumn(
        "ingest_timestamp",
        F.current_timestamp()
    )
)


# 6. Limpiar datos de aerolineas

airlines_bronze = (
    airlines_raw
    .withColumn(
        "airline_id",
        F.trim(F.col("airline_id").cast("string"))
    )
    .withColumn(
        "airline_name",
        F.trim(F.col("airline_name").cast("string"))
    )
    .filter(
        F.col("airline_id").isNotNull()
        & (F.col("airline_id") != "")
    )
    .withColumn(
        "ingest_timestamp",
        F.current_timestamp()
    )
)


# 7. Limpiar datos de aeropuertos

airports_bronze = (
    airports_raw
    .withColumn(
        "airport_id",
        F.trim(F.col("airport_id").cast("string"))
    )
    .withColumn(
        "city",
        F.trim(F.col("city").cast("string"))
    )
    .withColumn(
        "country",
        F.trim(F.col("country").cast("string"))
    )
    .filter(
        F.col("airport_id").isNotNull()
        & (F.col("airport_id") != "")
    )
    .withColumn(
        "ingest_timestamp",
        F.current_timestamp()
    )
)


# 8. Validar registros

datasets = [
    ("flights", flights_raw, flights_bronze),
    ("airlines", airlines_raw, airlines_bronze),
    ("airports", airports_raw, airports_bronze)
]

for nombre, original, limpio in datasets:
    total_original = original.count()
    total_limpio = limpio.count()

    print(
        f"{nombre}: {total_original} originales, "
        f"{total_limpio} limpios, "
        f"{total_original - total_limpio} eliminados"
    )


# 9. Guardar tablas Bronze

tablas_bronze = {
    "flights": flights_bronze,
    "airlines": airlines_bronze,
    "airports": airports_bronze
}

for nombre, dataframe in tablas_bronze.items():
    tabla_destino = f"{CATALOG}.{BRONZE_SCHEMA}.{nombre}"

    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabla_destino)
    )


# 10. Verificar resultados

for nombre in tablas_bronze:
    tabla = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{nombre}")
    print(f"{nombre}: {tabla.count()} registros guardados")


print("Proceso Bronze finalizado correctamente")


flights: 50000 originales, 49479 limpios, 521 eliminados
airlines: 6 originales, 6 limpios, 0 eliminados
airports: 10 originales, 10 limpios, 0 eliminados
flights: 49479 registros guardados
airlines: 6 registros guardados
airports: 10 registros guardados
Proceso Bronze finalizado correctamente



## Apartado 2. Transformaciones y modelado en Silver

En esta etapa se relacionan los datos de vuelos, aerolíneas y aeropuertos para obtener información más completa. También se calculan los retrasos, las distancias en kilómetros y los indicadores de puntualidad.

Finalmente, se revisan los valores nulos y duplicados antes de guardar las tablas en Silver.


In [0]:
# 1. Importar librerias
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

# 2. Configuracion
CATALOG = "trabajo_final_aerolineas"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

# 3. Cargar tablas Bronze
flights = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.flights")
airlines = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.airlines")
airports = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.airports")

# Revisar tipos de datos
for nombre, df in [("flights", flights), ("airlines", airlines), ("airports", airports)]:
    print(f"Esquema de {nombre}:")
    df.printSchema()

# 4. Preparar aeropuertos de origen y destino
origen = airports.select(
    F.col("airport_id").alias("origin_airport"),
    F.col("city").alias("origin_city"),
    F.col("country").alias("origin_country")
)

destino = airports.select(
    F.col("airport_id").alias("destination_airport"),
    F.col("city").alias("destination_city"),
    F.col("country").alias("destination_country")
)

# 5. Unir datos y calcular nuevas columnas
flights_silver = (
    flights
    .join(airlines.select("airline_id", "airline_name"), on="airline_id", how="left")
    .join(origen, on="origin_airport", how="left")
    .join(destino, on="destination_airport", how="left")
    .withColumn("delay_total", F.col("departure_delay") + F.col("arrival_delay"))
    .withColumn(
        "on_time",
        F.when(F.col("arrival_delay") <= 0, 1)
         .when(F.col("arrival_delay") > 0, 0)
    )
    .withColumn("distance_km", F.round(F.col("distance_miles") * 1.60934, 2))
    .withColumn(
        "delay_category",
        F.when(F.col("delay_total").isNull(), "Unknown")
         .when(F.col("delay_total") <= 0, "None")
         .when(F.col("delay_total") <= 15, "Short")
         .when(F.col("delay_total") <= 60, "Medium")
         .otherwise("Long")
    )
    .withColumn("flight_month", F.month("flight_date"))
    .withColumn("flight_quarter", F.quarter("flight_date"))
    .withColumn("flight_weekday", F.dayofweek("flight_date"))
    .dropDuplicates(["flight_id"])
)

# 6. Metricas por aerolinea
airline_metrics = (
    flights_silver
    .groupBy("airline_id", "airline_name")
    .agg(
        F.count("*").alias("total_flights"),
        F.round(F.avg("delay_total"), 2).alias("avg_delay"),
        F.round(F.sum("distance_km"), 2).alias("total_distance_km"),
        F.round(F.avg("on_time") * 100, 2).alias("on_time_pct")
    )
)

# 7. Metricas por aeropuerto de origen
origin_metrics = (
    flights_silver
    .groupBy("origin_airport", "origin_city", "origin_country")
    .agg(
        F.count("*").alias("departing_flights"),
        F.round(F.avg("departure_delay"), 2).alias("avg_departure_delay"),
        F.round(F.avg("on_time") * 100, 2).alias("on_time_pct")
    )
)

# 8. Metricas por aeropuerto de destino
destination_metrics = (
    flights_silver
    .groupBy("destination_airport", "destination_city", "destination_country")
    .agg(
        F.count("*").alias("arriving_flights"),
        F.round(F.avg("arrival_delay"), 2).alias("avg_arrival_delay"),
        F.round(F.avg("on_time") * 100, 2).alias("on_time_pct")
    )
)

# 9. Validar registros y valores faltantes
total_bronze = flights.count()
total_silver = flights_silver.count()

nulos_id = flights_silver.filter(F.col("flight_id").isNull() | (F.col("flight_id") == "")).count()
duplicados = (
    flights_silver.groupBy("flight_id").count()
    .filter(F.col("count") > 1).count()
)

columnas_importantes = [
    "flight_date", "airline_id", "airline_name", "origin_airport",
    "origin_city", "destination_airport", "destination_city",
    "departure_delay", "arrival_delay", "distance_miles"
]

nulos = flights_silver.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in columnas_importantes
]).first().asDict()

print(f"Vuelos Bronze: {total_bronze} | Silver: {total_silver} | ID nulos: {nulos_id} | Duplicados: {duplicados}")
print("Valores nulos por columna:", nulos)

if nulos_id > 0 or duplicados > 0:
    raise ValueError("Se encontraron problemas en los identificadores de vuelos")

# No se inventan fechas, retrasos ni datos de aeropuertos faltantes.
# Las funciones de promedio omiten los valores nulos.

# 10. Guardar tablas Silver
tablas_silver = {
    "flights_enriched": flights_silver,
    "airline_metrics": airline_metrics,
    "origin_airport_metrics": origin_metrics,
    "destination_airport_metrics": destination_metrics
}

for nombre, df in tablas_silver.items():
    tabla = f"{CATALOG}.{SILVER_SCHEMA}.{nombre}"
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabla)
    )

# 11. Verificar resultados
for nombre in tablas_silver:
    total = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.{nombre}").count()
    print(f"{nombre}: {total} registros")

print("Proceso Silver finalizado correctamente")


Esquema de flights:
root
 |-- flight_id: string (nullable = true)
 |-- flight_date: date (nullable = true)
 |-- airline_id: string (nullable = true)
 |-- origin_airport: string (nullable = true)
 |-- destination_airport: string (nullable = true)
 |-- departure_delay: integer (nullable = true)
 |-- arrival_delay: integer (nullable = true)
 |-- distance_miles: double (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)

Esquema de airlines:
root
 |-- airline_id: string (nullable = true)
 |-- airline_name: string (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)

Esquema de airports:
root
 |-- airport_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)

Vuelos Bronze: 49479 | Silver: 49479 | ID nulos: 0 | Duplicados: 0
Valores nulos por columna: {'flight_date': 0, 'airline_id': 0, 'airline_name': 0, 'origin_airport': 0, 'origin_city': 0, 'destination_ai


## Apartados 3 y 4. Análisis avanzado, métricas y capa Gold

En esta etapa se analizan los retrasos de los vuelos por mes, día de la semana y trimestre.

También se identifican los cinco vuelos con mayor retraso por aerolínea utilizando `row_number()` y `dense_rank()`. La primera función asigna una posición única a cada vuelo, mientras que la segunda permite identificar los empates.

Además, se clasifican los aeropuertos según su retraso promedio de salida y se calculan indicadores de puntualidad, cantidad de vuelos y retrasos por aerolínea y aeropuerto.

Finalmente, se generan las tablas Gold con los indicadores y rankings, y se aplica la optimización Delta para facilitar las consultas y el análisis de los resultados.


In [0]:
# 1. Importar librerias
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

# 2. Configuracion
CATALOG = "trabajo_final_aerolineas"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

# 3. Cargar datos de Silver
flights = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.flights_enriched")

# 4. Retrasos por mes, dia y trimestre
monthly_delays = (
    flights
    .groupBy(F.year("flight_date").alias("year"), F.month("flight_date").alias("month"))
    .agg(
        F.count("*").alias("total_flights"),
        F.round(F.avg("arrival_delay"), 2).alias("avg_arrival_delay"),
        F.round(F.avg("departure_delay"), 2).alias("avg_departure_delay"),
        F.round(F.avg("on_time") * 100, 2).alias("on_time_pct")
    )
)

weekday_delays = (
    flights.groupBy("flight_weekday")
    .agg(
        F.count("*").alias("total_flights"),
        F.round(F.avg("arrival_delay"), 2).alias("avg_arrival_delay")
    )
)

quarterly_delays = (
    flights
    .groupBy(F.year("flight_date").alias("year"), F.col("flight_quarter").alias("quarter"))
    .agg(
        F.count("*").alias("total_flights"),
        F.round(F.avg("arrival_delay"), 2).alias("avg_arrival_delay")
    )
)

# 5. Top 5 vuelos con mayor retraso por aerolinea
# row_number da posiciones unicas y dense_rank conserva los empates.
airline_window = (
    Window.partitionBy("airline_id")
    .orderBy(F.col("delay_total").desc_nulls_last(), F.col("flight_id").asc())
)

airline_rank_window = (
    Window.partitionBy("airline_id")
    .orderBy(F.col("delay_total").desc_nulls_last())
)

top_flights = (
    flights
    .withColumn("row_number", F.row_number().over(airline_window))
    .withColumn("delay_rank", F.dense_rank().over(airline_rank_window))
    .filter(F.col("row_number") <= 5)
    .select(
        "airline_id", "airline_name", "flight_id", "flight_date",
        "origin_airport", "destination_airport", "delay_total",
        "row_number", "delay_rank"
    )
)


# 6. Ranking de aeropuertos por retraso

# Calcular el retraso promedio de los vuelos salientes

airport_delays = (
    flights
    .groupBy("origin_airport", "origin_city")
    .agg(
        F.count("*").alias("departing_flights"),
        F.round(F.avg("departure_delay"), 2).alias("avg_delay"),
        F.round(F.avg("on_time") * 100, 2).alias("on_time_pct")
    )
)

# Ordenar los aeropuertos del mayor al menor retraso

airport_window = Window.orderBy(
    F.col("avg_delay").desc_nulls_last()
)

airport_ranking = (
    airport_delays
    .withColumn(
        "airport_rank",
        F.dense_rank().over(airport_window)
    )
)


# 7. Indicadores mensuales por aerolinea
airline_monthly_kpis = (
    flights
    .groupBy(
        "airline_id", "airline_name",
        F.year("flight_date").alias("year"),
        F.month("flight_date").alias("month")
    )
    .agg(
        F.count("*").alias("total_flights"),
        F.round(F.avg("arrival_delay"), 2).alias("avg_arrival_delay"),
        F.round(F.avg("delay_total"), 2).alias("avg_total_delay"),
        F.round(F.avg("on_time") * 100, 2).alias("on_time_pct")
    )
)

# 8. Indicadores mensuales por aeropuerto
airport_monthly_kpis = (
    flights
    .groupBy(
        "origin_airport", "origin_city",
        F.year("flight_date").alias("year"),
        F.month("flight_date").alias("month")
    )
    .agg(
        F.count("*").alias("departing_flights"),
        F.round(F.avg("departure_delay"), 2).alias("avg_departure_delay"),
        F.round(F.avg("on_time") * 100, 2).alias("on_time_pct")
    )
)

# 9. Indicadores generales por aerolinea
airline_kpis = (
    flights.groupBy("airline_id", "airline_name")
    .agg(
        F.count("*").alias("total_flights"),
        F.round(F.avg("arrival_delay"), 2).alias("avg_arrival_delay"),
        F.round(F.avg("delay_total"), 2).alias("avg_total_delay"),
        F.round(F.avg("on_time") * 100, 2).alias("on_time_pct"),
        F.round(F.sum("distance_km"), 2).alias("total_distance_km")
    )
)

# 10. Ranking de puntualidad de aeropuertos
punctuality_window = Window.orderBy(F.col("on_time_pct").desc_nulls_last())
airport_punctuality = (
    airport_delays
    .withColumn("punctuality_rank", F.dense_rank().over(punctuality_window))
)

# 11. Datos por vuelo para Machine Learning
# El retraso de llegada es el objetivo, no una variable de entrada.
ml_flights = (
    flights.select(
        "flight_id", "flight_date", "airline_id", "origin_airport",
        "destination_airport", "departure_delay", "distance_km",
        "flight_month", "flight_quarter", "flight_weekday", "arrival_delay"
    )
    .filter(F.col("arrival_delay").isNotNull())
)

# 12. Guardar tablas Gold
gold_tables = {
    "monthly_delays": monthly_delays,
    "weekday_delays": weekday_delays,
    "quarterly_delays": quarterly_delays,
    "top_flights_by_airline": top_flights,
    "airport_delay_ranking": airport_ranking,
    "airline_monthly_kpis": airline_monthly_kpis,
    "airport_monthly_kpis": airport_monthly_kpis,
    "airline_kpis": airline_kpis,
    "airport_punctuality_ranking": airport_punctuality,
    "ml_flights": ml_flights
}

for nombre, df in gold_tables.items():
    tabla = f"{CATALOG}.{GOLD_SCHEMA}.{nombre}"
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabla)
    )

# 13. Optimizar tablas Delta
for nombre in gold_tables:
    spark.sql(f"OPTIMIZE {CATALOG}.{GOLD_SCHEMA}.{nombre}")

# 14. Verificar resultados
for nombre in gold_tables:
    total = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{nombre}").count()
    print(f"{nombre}: {total} registros")

print("Proceso Gold finalizado correctamente")


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


monthly_delays: 6 registros
weekday_delays: 7 registros
quarterly_delays: 2 registros
top_flights_by_airline: 30 registros
airport_delay_ranking: 10 registros
airline_monthly_kpis: 36 registros
airport_monthly_kpis: 60 registros
airline_kpis: 6 registros
airport_punctuality_ranking: 10 registros
ml_flights: 49244 registros
Proceso Gold finalizado correctamente


### 3.1 Análisis temporal de retrasos

In [0]:

# Revisar los cinco vuelos con mayor retraso por aerolinea
display(spark.table("trabajo_final_aerolineas.gold.top_flights_by_airline").orderBy("airline_id", "row_number"))

# Revisar el ranking de aeropuertos
display(spark.table("trabajo_final_aerolineas.gold.airport_delay_ranking").orderBy("airport_rank"))

# Revisar los indicadores mensuales por aerolinea
display(spark.table("trabajo_final_aerolineas.gold.airline_monthly_kpis").orderBy("airline_id", "year", "month"))


airline_id,airline_name,flight_id,flight_date,origin_airport,destination_airport,delay_total,row_number,delay_rank
AA,American Airlines,FL-114276,2025-06-29,SEA,LAS,164,1,1
AA,American Airlines,FL-106973,2025-06-05,DEN,SEA,154,2,2
AA,American Airlines,FL-125098,2025-03-31,ORD,DEN,154,3,2
AA,American Airlines,FL-148219,2025-06-16,SFO,ORD,149,4,3
AA,American Airlines,FL-128580,2025-03-24,DEN,SEA,145,5,4
B6,JetBlue Airways,FL-139444,2025-01-14,SFO,DEN,199,1,1
B6,JetBlue Airways,FL-121487,2025-01-23,SFO,ATL,179,2,2
B6,JetBlue Airways,FL-122541,2025-03-14,SFO,MIA,178,3,3
B6,JetBlue Airways,FL-122207,2025-05-19,DEN,ATL,175,4,4
B6,JetBlue Airways,FL-100239,2025-02-10,MIA,ATL,170,5,5


origin_airport,origin_city,departing_flights,avg_delay,on_time_pct,airport_rank
DFW,Dallas,4915,8.85,27.61,1
LAS,Las Vegas,4962,8.82,28.0,2
ORD,Chicago,5007,8.52,28.65,3
SFO,San Francisco,4929,8.39,28.82,4
LAX,Los Angeles,4965,8.38,29.09,5
SEA,Seattle,4891,8.33,28.29,6
MIA,Miami,4878,8.3,28.67,7
ATL,Atlanta,5010,8.14,29.02,8
JFK,New York,4899,8.12,28.56,9
DEN,Denver,5023,8.06,29.95,10


airline_id,airline_name,year,month,total_flights,avg_arrival_delay,avg_total_delay,on_time_pct
AA,American Airlines,2025,1,1501,10.81,16.43,31.21
AA,American Airlines,2025,2,1316,9.63,14.72,33.77
AA,American Airlines,2025,3,1426,8.6,12.69,36.1
AA,American Airlines,2025,4,1409,9.56,14.27,33.9
AA,American Airlines,2025,5,1372,10.07,15.22,34.31
AA,American Airlines,2025,6,1407,10.46,15.69,34.24
B6,JetBlue Airways,2025,1,1363,20.55,35.73,19.14
B6,JetBlue Airways,2025,2,1282,19.3,33.83,20.72
B6,JetBlue Airways,2025,3,1440,19.62,34.06,20.13
B6,JetBlue Airways,2025,4,1330,19.82,34.4,18.25



%md

## Apartado 5. Jobs y Workflows

Para automatizar el procesamiento de los datos, se creó el Job `Trabajo_Final_Aerolineas_Pipeline` en Databricks, compuesto por tres tareas: Bronze_Ingestion, Silver_Transform y Gold_Analytics.

Las tareas se configuraron para ejecutarse en orden, con dependencias, reintentos y notificaciones por correo en caso de error.

El flujo se ejecutó correctamente y las tres tareas finalizaron con estado `Succeeded`.

La configuración del Job se entrega en un archivo YAML, junto con los tres archivos Python y las capturas de pantalla que evidencian la configuración y ejecución del Workflow.
